In [3]:
from langgraph.graph import StateGraph

# nodes
def node_a(state):
    return {"msg":"Hello from A"}

def node_b(state):
    return {"msg": state["msg"] + " -> B"}

# Graph
graph = StateGraph(dict)

graph.add_node("A", node_a)
graph.add_node("B", node_b)

graph.set_entry_point("A")

# Normal Edge
graph.add_edge("A","B")

app = graph.compile()
print(app.invoke({}))

{'msg': 'Hello from A -> B'}


In [7]:
from langgraph.graph import StateGraph

# nodes
def router(state):
    if "math" in state["input"]:
        return {"route": "tool"}
    return {"route": "llm"}

def tool_node(state):
    return {"output": "Using Calculator"}

def llm_node(State):
    return {"output": "Using LLM"}

# Graph
graph = StateGraph(dict)

graph.add_node("router", router)
graph.add_node("tool", tool_node)
graph.add_node("llm", llm_node)

graph.set_entry_point("router")

# Conditional edge
graph.add_conditional_edges(
    "router",
    lambda state: state["route"],
    {
        "tool": "tool",
        "llm": "llm"
    }
)

app = graph.compile()

print(app.invoke({"input":"solve math problem"}))

{'output': 'Using Calculator'}


In [11]:
from typing import TypedDict
from langgraph.graph import StateGraph

# Define State
class state(TypedDict, total = False):
    input: str
    tool: str
    log: str
    result: str

# Nodes
def router(state):
    return {}

def tool_node(state):
    return {"tool": "Tool executed"}

def logger_node(state):
    return {"log": "Logged input"}

def final_node(state):
    return {
        "result": f"{state.get('tool')} | {state.get('log')}"
    }

# Build Graph

graph = StateGraph(state)

graph.add_node("router", router)
graph.add_node("tool", tool_node)
graph.add_node("logger", logger_node)
graph.add_node("final", final_node)

graph.set_entry_point("router")

# Parallel routing

graph.add_conditional_edges(
    "router",
    lambda state: ["tool", "logger"],
    {
        "tool":"tool",
        "logger": "logger"
    }
)

# merge (fan-in)
graph.add_edge("tool", "final")
graph.add_edge("logger", "final")

# Compile And Run

app = graph.compile()

result = app.invoke({"input":"hello"})
print(result)

{'input': 'hello', 'tool': 'Tool executed', 'log': 'Logged input', 'result': 'Tool executed | Logged input'}
